# Environment Setup and the Running Business Scenario

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Installation

In [ ]:
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

Package roles:

| Package | Used for |
|---|---|
| `statsmodels` | OLS, GLS, segmented regression, ARIMA, diagnostics |
| `linearmodels` | Panel fixed effects (`PanelOLS`), IV (`IV2SLS`) |
| `tfp-causalimpact` | Bayesian structural time series (Google's CausalImpact, TFP port) |
| `pysyncon` | Synthetic control (Abadie-style, plus augmented/penalized variants) |
| `rdrobust` | Robust regression discontinuity estimation and bandwidth selection |
| `doubleml` | Double/debiased machine learning |
| `econml` | Heterogeneous treatment effects (DML, DR-learner, causal forests) |
| `dowhy` | DAG-based identification, refutation tests |
| `scikit-learn` | Nuisance models (propensity scores, outcome models) |

> **Note on CausalImpact ports:** the original `pycausalimpact` package is unmaintained; `tfp-causalimpact` (TensorFlow Probability) is the actively maintained port and is what Google recommends. The API shown in Section 7 follows `tfp-causalimpact`; if you use a different port, the constructor arguments differ slightly but the workflow is identical.

## Common imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## The running scenario: *NorthShore Retail*

To keep the tutorial concrete, most sections use variations of a single simulated company:

> **NorthShore Retail** operates 120 stores across 12 regional markets. In June 2024 the company launched a **"Fresh Local" store remodel + marketing program** in some markets. Leadership wants to know the program's causal effect on weekly revenue. Different sections will vary *how* the program was assigned (by market, by store eligibility score, by staggered rollout, by store-manager self-selection) — because the assignment mechanism is exactly what determines which method is valid.

Using simulated data has a pedagogical advantage: **we know the true effect**, so we can check whether each estimator recovers it and watch naive approaches fail.

## Master data-generating function

The generator below is reused (with modifications) throughout. It builds weekly store-level panel data with store fixed effects, market-level shocks, seasonality, a secular trend, and a known treatment effect.

In [7]:


# The function below simulates a longitudinal (panel) dataset mimicking the NorthShore Retail scenario.
# This synthetic data allows us to know the "true" treatment effect for later causal inference checks.

def make_panel(
    n_stores=120,                # total number of stores in simulation
    n_markets=12,                # total number of geographic markets
    n_weeks=104,                 # number of time periods (weeks)
    treat_markets=(0, 1, 2, 3),  # market indices where the program will be introduced
    treat_week=60,               # week when the program launches ("intervention")
    true_effect=8.0,             # full treatment effect on revenue (in $k)
    effect_ramp=6,               # treatment ramps up to full effect over these weeks
    seed=42,                     # random seed for reproducibility
):
    # Set up a reproducible random generator.
    rng = np.random.default_rng(seed)

    # Assign each store to a market (even distribution).
    stores = np.arange(n_stores)
    markets = stores % n_markets

    # Randomly generate:
    # - Store fixed effects (baseline revenue per store)
    # - Market fixed effects (shared environment for all stores in a market)
    # - Store-level linear revenue trends (some stores naturally grow faster)
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    # Simulate market-specific shocks (arising each week, same for all stores in a market)
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        # Simulate seasonal effects (peaks in certain weeks)
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]        # market index for this store
            # Identify if this unit will ever be treated (group) and if we are post-launch (time)
            treated_unit = m in treat_markets
            post = w >= treat_week
            # Treatment ramps in over effect_ramp weeks.
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp    # rising effect post-intervention
            else:
                tau = 0.0
            # Compose the outcome:
            # Sum of: store+market revenue, store trend, seasonality, market shock, 
            # treatment effect (if any), and independent noise.
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            # Store a row with:
            # [store id, market id, week, treated indicator, post indicator, D=treatment active, simulated revenue]
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    # Collect all records into a DataFrame with descriptive columns:
    # - "store"     : store ID
    # - "market"    : market ID
    # - "week"      : week number
    # - "treated"   : 1 if this store *ever* treated; 0 otherwise
    # - "post"      : 1 if in/after treatment week; 0 otherwise
    # - "D"         : 1 if both treated & post; 0 otherwise (actual treatment status)
    # - "revenue"   : simulated observed outcome (in $k)
    df = pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])
    return df

# Example usage:
# Generate one simulated panel using default parameters, and show its top rows.
panel = make_panel()
panel.head()

,store,market,week,treated,post,D,revenue
0,0,0,0,1,0,0,99.304
1,1,1,0,1,0,0,84.039
2,2,2,0,1,0,0,119.259
3,3,3,0,1,0,0,129.004
4,4,4,0,0,0,0,75.000


Key columns:

- `treated` — 1 if the store belongs to a market that *ever* receives the program (group indicator).
- `post` — 1 for weeks at or after launch (time indicator).
- `D = treated × post` — the actual treatment status in each store-week. The coefficient on `D` is the causal effect we want.
- True effect: **+$8k/week at full ramp**.